# LR Finder and Deliberate Overfitting Check

Companion notebook for the [Practical Training Decisions](https://codemia.ai/courses/deep_learning_foundations/practical_training_decisions) lesson on Codemia.

**What you will build**: two debugging tools every practitioner should have in muscle memory — Smith's learning rate finder, and the 60-second deliberate-overfit sanity check that catches silent training bugs before you waste hours.

**Time**: ~30 minutes

**GPU required**: no — everything in this notebook runs on a CPU in a few minutes. That is the point: these are checks you should run *before* you spin up a GPU.

## Setup

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')
# CPU is fine — we deliberately keep the model and data tiny.

## Concept recap

Two of the highest-leverage debugging tools you can have in muscle memory:

**LR finder** (Smith, 2017). The single hyperparameter that matters most is the learning rate. Rather than guess, sweep it: train one minibatch at a time while *exponentially increasing* the LR from something tiny (1e-7) to something huge (10). Plot loss vs. log-LR. The curve has three regions — flat (LR too small to learn), steep descent (the sweet spot), explosion (LR too large, loss diverges). Pick an LR roughly 10x lower than where the steepest descent happens, or where the slope is most negative. Two minutes of LR-finder beats hours of hand tuning.

**Deliberate overfitting check** (the lesson's Section 4). A correct training loop *must* be able to overfit a tiny batch (10 samples) to near-zero loss in under 100 steps. If it can't, something upstream is broken: wrong loss reduction, missing `zero_grad`, model frozen in eval mode, learning rate effectively zero. Running this check before a real training run takes under a minute on CPU and catches silent bugs that would otherwise eat a day.

## Implementation

### Step 1 — Tiny ConvNet on MNIST

We use MNIST (not CIFAR-10) so the whole notebook runs on CPU in minutes. The architecture is intentionally small: two conv blocks then a linear head. Real models are bigger; the *diagnostics* we are practicing scale to any size.

In [ ]:
class TinyConvNet(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        # Two conv blocks with pooling — enough capacity to learn MNIST,
        # small enough to train on CPU.
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 28 -> 14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 14 -> 7
        )
        # Dropout is here on purpose — Variation 3 will exploit train/eval mismatch.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = TinyConvNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'param count: {n_params:,}')

In [ ]:
# Standard MNIST normalization. We subset to ~6000 training images so the LR finder
# and short training runs both finish quickly on CPU.
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
full_train = datasets.MNIST('./data', train=True, download=True, transform=tfm)
subset_idx = torch.randperm(len(full_train))[:6000].tolist()
train_ds = Subset(full_train, subset_idx)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
print(f'train samples: {len(train_ds)}, batches: {len(train_loader)}')

### Step 2 — The LR finder

Walk the LR up by a fixed multiplicative factor every minibatch. Record loss at each step. We stop early if the loss explodes (>4x its initial value) — once the model diverges, further data is noise.

In [ ]:
def lr_finder(model, loader, start=1e-7, end=10.0, num_iter=200):
    # Snapshot weights so we can restore — the LR finder *destroys* the model on purpose.
    snapshot = copy.deepcopy(model.state_dict())
    opt = torch.optim.SGD(model.parameters(), lr=start)
    # Multiplicative step: r ** num_iter takes us from start to end.
    mult = (end / start) ** (1.0 / num_iter)
    lrs, losses = [], []
    best = float('inf')
    it = iter(loader)
    model.train()
    for step in range(num_iter):
        try:
            xb, yb = next(it)
        except StopIteration:
            it = iter(loader); xb, yb = next(it)
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        opt.step()
        cur_lr = opt.param_groups[0]['lr']
        lrs.append(cur_lr); losses.append(loss.item())
        best = min(best, loss.item())
        # Bail once the loss diverges — noisy data past this point distorts the plot.
        if loss.item() > 4 * best or math.isnan(loss.item()):
            print(f'  diverged at step {step}, lr={cur_lr:.2e}'); break
        for g in opt.param_groups:
            g['lr'] *= mult
    # Restore the model so the user can call lr_finder repeatedly.
    model.load_state_dict(snapshot)
    return lrs, losses

In [ ]:
torch.manual_seed(0)
model = TinyConvNet().to(device)
lrs, losses = lr_finder(model, train_loader, num_iter=200)
print(f'recorded {len(lrs)} steps; min loss = {min(losses):.4f} at lr = {lrs[losses.index(min(losses))]:.2e}')

### Step 3 — Pick the LR

The classic heuristic: find the point of *steepest descent* (most negative slope on the log-LR axis), then take roughly 10x lower. The min-loss point itself is too aggressive — the model is already on the verge of diverging there.

In [ ]:
# Smooth losses with a small window so we pick the elbow on the trend, not on a noisy spike.
def smooth(xs, beta=0.9):
    out, m = [], xs[0]
    for x in xs:
        m = beta * m + (1 - beta) * x; out.append(m)
    return out

smoothed = smooth(losses, beta=0.85)
log_lrs = [math.log10(l) for l in lrs]
# Slope = d(loss) / d(log-lr), via finite difference. Most negative slope = fastest descent.
slopes = [smoothed[i+1] - smoothed[i] for i in range(len(smoothed) - 1)]
i_steep = min(range(len(slopes)), key=lambda i: slopes[i])
lr_steep = lrs[i_steep]
lr_suggest = lr_steep / 10.0
print(f'steepest descent at lr = {lr_steep:.2e}')
print(f'suggested lr (10x lower)  = {lr_suggest:.2e}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(lrs, smoothed, label='loss (smoothed)')
plt.axvline(lr_steep, color='orange', ls='--', label=f'steepest @ {lr_steep:.1e}')
plt.axvline(lr_suggest, color='green', ls='--', label=f'suggested @ {lr_suggest:.1e}')
plt.xscale('log')
plt.xlabel('learning rate (log)'); plt.ylabel('loss')
plt.title('LR finder: loss vs LR'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## Experiment — train at the suggested LR, vs too-low and too-high

Same model, same data, three learning rates. Watch how badly the wrong LR performs.

In [ ]:
def train_short(lr, epochs=2):
    torch.manual_seed(0)
    m = TinyConvNet().to(device)
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    losses = []
    m.train()
    for ep in range(epochs):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(m(xb), yb)
            loss.backward(); opt.step()
            losses.append(loss.item())
        print(f'  lr={lr:.1e}  epoch {ep}  last batch loss={losses[-1]:.4f}')
    return losses

h_low     = train_short(lr=1e-6)
h_suggest = train_short(lr=lr_suggest)
h_high    = train_short(lr=1.0)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(smooth(h_low, 0.9),     label='too low (1e-6)')
plt.plot(smooth(h_suggest, 0.9), label=f'suggested ({lr_suggest:.1e})')
plt.plot(smooth(h_high, 0.9),    label='too high (1.0)')
plt.xlabel('step'); plt.ylabel('loss (smoothed)')
plt.title('Training loss at three learning rates'); plt.legend(); plt.grid(alpha=0.3); plt.show()

Three different stories on one plot: the too-low LR barely moves, the too-high LR oscillates / diverges, and the suggested LR lands in the steep-descent region. The LR finder pays for itself the first time you use it.

## Implementation — the deliberate overfitting check

Take a *single* tiny batch (10 samples). Re-run the same train step on it repeatedly. A correct training loop should drive the loss to ~0 within a hundred or so steps — there is nothing else to fit. If it doesn't, your training step is broken.

In [ ]:
def overfit_check(train_step_fn, batch, steps=200, target_loss=1e-3, log_every=20):
    # train_step_fn(model, opt, xb, yb) -> loss (Python float).
    # We re-create model+opt internally so each call is independent.
    torch.manual_seed(0)
    m = TinyConvNet().to(device)
    opt = torch.optim.SGD(m.parameters(), lr=1e-2)
    xb, yb = batch
    history = []
    for step in range(steps):
        loss = train_step_fn(m, opt, xb, yb)
        history.append(loss)
        if step % log_every == 0 or step == steps - 1:
            print(f'  step {step:3d}  loss={loss:.6f}')
        if loss < target_loss:
            print(f'  reached target loss {target_loss:g} at step {step}')
            return history, True
    return history, False

# Grab one tiny fixed batch and freeze it on the device.
tiny_xb, tiny_yb = next(iter(DataLoader(train_ds, batch_size=10, shuffle=True)))
tiny_batch = (tiny_xb.to(device), tiny_yb.to(device))

## Bug demonstration — wrong loss reduction

A real bug seen in the wild: someone copy-pastes a manual loss computation, uses `reduction='sum'`, then divides by the *full dataset size* instead of the batch size. The gradient is silently scaled down by ~1000x. The model trains, the loss decreases, but it crawls. This is exactly the kind of thing the overfitting check catches in seconds.

In [ ]:
def buggy_train_step(model, opt, xb, yb):
    opt.zero_grad()
    # WRONG: reduction='sum' divided by 60000 (the full MNIST train size, not the batch size).
    # The gradient is shrunk by ~6000x relative to what the optimizer expects, so steps
    # are effectively LR=1e-2 / 6000 = 1.6e-6. Training appears to work but at a glacial pace.
    loss = F.cross_entropy(model(xb), yb, reduction='sum') / 60000
    loss.backward(); opt.step()
    return loss.item()

def correct_train_step(model, opt, xb, yb):
    opt.zero_grad()
    # Correct: default reduction='mean' divides by the batch size automatically.
    loss = F.cross_entropy(model(xb), yb)
    loss.backward(); opt.step()
    return loss.item()

print('=== Buggy step (reduction=sum / wrong denominator) ===')
h_buggy, ok_buggy = overfit_check(buggy_train_step, tiny_batch, steps=200)
print(f'  buggy succeeded? {ok_buggy}\n')

print('=== Correct step ===')
h_correct, ok_correct = overfit_check(correct_train_step, tiny_batch, steps=200)
print(f'  correct succeeded? {ok_correct}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(h_buggy, label='buggy (wrong reduction)')
plt.plot(h_correct, label='correct')
plt.axhline(1e-3, color='gray', ls=':', label='target')
plt.yscale('log')
plt.xlabel('step'); plt.ylabel('loss (log)')
plt.title('Overfitting check: bug vs. correct'); plt.legend(); plt.grid(alpha=0.3); plt.show()

The visual is unambiguous: the correct step plunges to the target inside 200 steps, the buggy step plateaus orders of magnitude above it. You don't need to read the code carefully to know something is wrong.

## Variations — two more bugs the overfitting check catches

### Variation 1 — missing `optimizer.zero_grad()`

PyTorch *accumulates* gradients across `.backward()` calls by default. Forgetting `zero_grad` means the gradient at step $t$ is the sum of gradients from steps $0..t$. The effective LR explodes; training diverges or oscillates wildly.

In [ ]:
def step_no_zero_grad(model, opt, xb, yb):
    # WRONG: no opt.zero_grad() — gradients pile up across steps and the effective LR explodes.
    loss = F.cross_entropy(model(xb), yb)
    loss.backward(); opt.step()
    return loss.item()

print('=== Missing zero_grad ===')
h_nograd, ok_nograd = overfit_check(step_no_zero_grad, tiny_batch, steps=200)
print(f'  succeeded? {ok_nograd}')

### Variation 2 — left in `model.eval()`

Dropout layers behave differently in train vs. eval mode. If you `model.eval()` to inspect predictions and forget to `model.train()` afterwards, your *training* loop runs with dropout disabled and BatchNorm using running stats. On a tiny batch the consequences are subtle but real — and the overfitting check still catches it because the train/eval mismatch breaks gradient flow through dropout in characteristic ways.

In [ ]:
def step_stuck_in_eval(model, opt, xb, yb):
    # WRONG: model is left in eval mode. With this small a network it still trains,
    # but compare the curve shape against the correct run. A more dropout-heavy
    # model would show a much larger gap.
    model.eval()
    opt.zero_grad()
    loss = F.cross_entropy(model(xb), yb)
    loss.backward(); opt.step()
    return loss.item()

print('=== Stuck in eval mode ===')
h_eval, ok_eval = overfit_check(step_stuck_in_eval, tiny_batch, steps=200)
print(f'  succeeded? {ok_eval}')

In [ ]:
# Plot all four side by side. The correct run is the only one that hits the target cleanly.
plt.figure(figsize=(8, 4.5))
plt.plot(h_correct, label='correct')
plt.plot(h_buggy,   label='wrong reduction')
plt.plot(h_nograd,  label='missing zero_grad')
plt.plot(h_eval,    label='stuck in eval')
plt.axhline(1e-3, color='gray', ls=':', label='target')
plt.yscale('log')
plt.xlabel('step'); plt.ylabel('loss (log)')
plt.title('Overfitting check catches three different bugs')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## What to try next

- Wire `overfit_check(correct_train_step, tiny_batch, steps=200)` into a unit test in your repo. Any PR that breaks the train step now fails CI in a few seconds rather than during a 6-hour training run.
- Combine the LR finder with Smith's *OneCycle* schedule (`torch.optim.lr_scheduler.OneCycleLR`) to get "superconvergence" — train to a target accuracy in a fraction of the usual epoch count.
- Re-run the LR finder with `torch.optim.Adam` instead of SGD. Adam's effective step size is much less LR-sensitive; the curve looks different.
- Add a fourth bug to the variations: detach the model output (`.detach()`) before computing the loss. Watch the gradient become identically zero and the overfitting check report a flat line.